# Práctico 2 - Aplicación de LLMs


Para este práctico, se pide:

1. Cargar un SLM (Small Language Model) y hacerlo funcionar a modo de pregunta respuesta como lo haria ChatGPT -> Una opcion puede ser [Qwen de Alibaba](https://huggingface.co/Qwen/Qwen3-0.6B) u otro modelo similar con < 1 Bi de parámetros.

1. Cargar el dataset elegido en el Práctico 1 e iterar cada una de las filas para generar una predicción.

1. Cargar el modelo entrenado en el Práctico 1 y hacer una comparación ¿Es mejor el LLM para clasificar? ¿Por qué?

1. Iterar parametros y prompt para ver como mejora.

1. Finetunear el SLM (Opcional).

1. Incorporar un modelo de huggingface (ej. BERT) a la comparación (Opcional).

La notebook a presentar debe ser legible incluyendo.
1. Introducción

1. Acompañar con comentarios que aporten a la interpretación de los resultados.

1. Una conclusión (breve pero no tan breve) con un resumen de lo trabajado y los resultados más representativos de acuerdo a su interpretación


In [1]:
import json
import requests

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split


## Preparamos el terreno

In [2]:
!pip install transformers

Importamos e instanciamos

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# Cargar tokenizer y modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Prompt estilo chat
messages = [
    {"role": "user", "content": "Dame una breve introducción a los modelos de lenguaje grandes."}
]

# Aplicar plantilla de chat
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Tokenizar entrada
inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generar respuesta
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,# En false es deterministico
    top_p=0.9,
    temperature=0.7
)

# Decodificar solo la parte nueva generada
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("Assistant:", response)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Assistant: <think>
Okay, the user wants a brief introduction to large language models. Let me start by recalling what I know about them. First, they're big language models, so I need to mention their size and capabilities. I should explain their training data and how they process text. Maybe compare them to each other to highlight differences. Also, mention their applications like chatbots, content generation, etc. Make sure it's concise but covers the key points without getting too technical.
</think>

Los modelos de lenguaje grande son sistemas de inteligencia artificial que son entrenados con grandes conjuntos de textos para generar, traducir, analizar y responder a preguntas complejas. Estos modelos se entrenan con datos de múltiples lenguajes y tienen capacidad para procesar lenguaje natural de manera eficiente. Por ejemplo, modelos como GPT o BERT son conocidos por su capacidad para interactuar con usuarios de manera natural y ofrecer respuestas precisas. Aunque varían en tamaño 

## SLM QA con Qwen <1B



Usaremos un SLM <1B en Colab T4 y lo pondremos a funcionar en modo pregunta-respuesta tipo chat.

In [4]:
!nvidia-smi

!pip -q install "transformers>=4.44" accelerate bitsandbytes

Sun Oct 19 23:28:41 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             28W /   70W |    1608MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

### Elegir un SLM

Optaremos por un modelo seguro: "Qwen/Qwen3-0.6B" (chat simple, 0.6B)

1.   Elemento de la lista
2.   Elemento de la lista



In [6]:
model_id = "Qwen/Qwen3-0.6B"

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
tok = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=dtype,
)

### Función de “chat” estilo ChatGPT

Usamos la plantilla de chat nativa del tokenizer. Parametrizamos generación para control de estilo y determinismo.

In [7]:
# función 'chat' imita el comportamiento de ChatGPT

def chat(system:str, user:str,
         max_new_tokens=256,
         temperature=0.7,
         top_p=0.9,
         do_sample=True):
    """
    Retorna solo la respuesta del asistente.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user})

    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample, # En false es deterministico
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tok.eos_token_id
        )

    # recortamos la parte nueva generada
    gen = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return gen.strip()


Hacemos una prueba rápida:

In [8]:
resp = chat(
    system="Eres un asistente conciso y claro.",
    user="Dame una breve introducción a los modelos de lenguaje grandes."
)
print(resp)


<think>
Okay, the user wants a short introduction to large language models. Let me start by recalling what I know. Large language models are big in size, so I should mention their size, like terabytes or petabytes. They're trained on huge datasets. I need to highlight their capabilities, such as understanding complex text and generating coherent responses. Also, mention the training process, maybe how they're trained on millions of words. Emphasize their applications in different fields. Keep it concise, avoid jargon, and make sure it's clear and to the point.
</think>

Los modelos de lenguaje grande son inteligentes sistemas de aprendizaje automático entrenados en grandes conjuntos de texto para generar respuestas complejas y entender lenguaje humano. Se entrenan en datos de millones de palabras y son aplicados en áreas como inteligencia artificial, traducción, y servicios de soporte.


¿Qué sucede si bajamos la temperatura? Probamos un modo mas determinísta.

In [9]:
resp_det = chat(
    system="Eres un asistente técnico. Responde en 3 oraciones.",
    user="Explica qué es atención (self-attention) en 3 oraciones.",
    max_new_tokens=180,
    temperature=0.0,
    top_p=0.9, # mantenemos top_p
    do_sample=False
)
print(resp_det)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<think>
Okay, the user wants me to explain self-attention in three sentences. Let me start by recalling what self-attention is. It's a technique in machine learning where the model focuses on certain parts of the input to produce the best output. I should mention how it's used in tasks like language processing. Then, I need to explain the mechanism, maybe how it's implemented with weights. Finally, highlight its benefits like improving performance. Let me check if I'm covering all key points without being too technical. Yeah, that should work.
</think>

La atención (self-attention) es un mecanismo en el que el modelo de machine learning enfoca en partes específicas del texto o la imagen para producir una respuesta más precisa. Se basa en la idea de que cada parte del input tiene un peso diferente según su relevancia. Esto permite que el modelo priorice


La advertencia sobre temperature y top_p es normal: algunos modelos Qwen ignoran esos parámetros y se comportan de modo casi determinista igual.

Vamos con un ejemplo más estructurado.

En 'system' se define el rol (estilo, tono, formato) y tambien se puede pedir longitud y formato de salida explícitos.

In [10]:
resp_struct = chat(
    system="Eres un analista. Devuelve exactamente un JSON válido.",
    user='Resume en 2 bullets el concepto de “overfitting” y añade {"term":"overfitting"} como campo.',
    max_new_tokens=120,
    temperature=0.0,
    do_sample=False
)
print(resp_struct)

<think>
Okay, the user wants me to summarize the concept of "overfitting" in two bullets and include the term "overfitting" as a field. Let me start by recalling what overfitting is. It's when a model learns too much from the training data, leading to poor generalization. So the first bullet should mention that it's about learning too much and not generalizing well. The second bullet could explain the consequences, like increased error rates. Then, I need to make sure the JSON structure is correct. The term "overfitting" needs to be a


De esta manera:

* Se cargó un SLM de < 1 B (Qwen 0.6 B).

* Se implementó comunicación tipo chat con prompts de usuario y sistema.

* Se validó su capacidad de respuesta determinista y control por parámetros.

Como conclusión de esta sección, se puede decir que bajar temperature a 0 reduce la variabilidad, pero en Qwen 0.6 B la generación sigue siendo muy estable incluso con sampling activo. El modelo devuelve respuestas consistentes y completas en lenguaje natural.

## Usar el SLM como clasificador

La idea será cargar el dataset del Práctico 1 y hacer que Qwen clasifique cada texto mediante prompting. Después podremos comparar con el modelo usado en el TP1.

### Preparamos el entorno

In [11]:
import pandas as pd
import re
from sklearn.metrics import classification_report
from tqdm import tqdm

### Cargamos el dataset 'Reviews IMDB'

In [12]:
# Remueve archivo anterior (si existe)
!rm -f imdb_dataset.csv

# Usa wget para descarga
file_id = "1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3"
!wget --no-check-certificate "https://docs.google.com/uc?export=download&id={file_id}" -O imdb_dataset.csv

# Carga el dataset
with open("imdb_dataset.csv", 'r') as file:
    df_imdb = pd.read_csv(file)
    print(f"Success! Dataset loaded with {len(df_imdb)} records.")

--2025-10-19 23:31:14--  https://docs.google.com/uc?export=download&id=1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3
Resolving docs.google.com (docs.google.com)... 74.125.130.139, 74.125.130.100, 74.125.130.113, ...
Connecting to docs.google.com (docs.google.com)|74.125.130.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3&export=download [following]
--2025-10-19 23:31:14--  https://drive.usercontent.google.com/download?id=1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 64.233.170.132, 2404:6800:4003:c1a::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|64.233.170.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66212309 (63M) [application/octet-stream]
Saving to: ‘imdb_dataset.csv’

imdb_dataset.csv    100%[===================>]  63.14M   201MB/s   

In [13]:
df = df_imdb
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### Definimos prompt de clasificación

Diseñamos una plantilla que fuerce salida estructurada label=....

Ejemplo binario “positive/negative”

In [14]:
def classify_review(text):
    system_prompt = (
        "Eres un modelo de análisis de sentimientos. "
        "Responde SOLAMENTE con una de estas dos palabras, en minúsculas: positive o negative. "
        "No agregues explicación ni ningún otro texto."
    )
    user_prompt = f"Review:\n\"\"\"{text}\"\"\"\nEtiqueta:"
    out = chat(
        system_prompt,
        user_prompt,
        max_new_tokens=4,
        temperature=0.0,
        do_sample=False,
    )
    # limpiar espacios y tags
    out = re.sub(r"<.*?>", "", out).strip().lower()
    if out.startswith("positive"):
        return "positive"
    if out.startswith("negative"):
        return "negative"
    return "unk"

### Aplicar fila a fila y evaluar

Por limitaciones computacionales (se estimaba 11 hs para correr las cincuenta mil filas), se evaluó sobre una muestra estratificada de 500 reseñas balanceadas.


In [15]:
df_sample = df.groupby("sentiment", group_keys=False).apply(lambda x: x.sample(250, random_state=0))
print(df_sample["sentiment"].value_counts())

sentiment
negative    250
positive    250
Name: count, dtype: int64


/tmp/ipython-input-1933976793.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby("sentiment", group_keys=False).apply(lambda x: x.sample(250, random_state=0))


Antes de correr las 500 filas, haremos una prueba pequeña para ver si funciona:

In [16]:
test_df = df_sample.head(5).copy()

test_preds = []
for _, row in test_df.iterrows():
    pred = classify_review(row["review"])
    test_preds.append(pred)
    print(f"\nPred: {pred}\nTexto: {row['review'][:200]}...\n")

test_df["pred_llm"] = test_preds
print(test_df[["sentiment","pred_llm"]])


Pred: unk
Texto: I am wanting to make a "Holmes with Doors" pun but I can't quite string it all together. Suitably grubby and over edited WONDERLAND gives Kilmer a role that channels Morrison at the same time....but h...


Pred: unk
Texto: This had high intellectual pretensions.The main lead intends to give a "deep" "meaningful" rendering(with voice over for his frames of mind naturally) and he was certainly influenced by the fifties/si...


Pred: unk
Texto: Tim Robbins is oddly benign here, cast as a garage mechanic in 1950s New Jersey who falls in love with a perky blonde who turns out to be Albert Einstein's niece! Although he's on-screen much of the t...


Pred: unk
Texto: Poorly acted, poorly written and poorly directed. Special effects are cheap. Best performance is by Yvette Napir, but that's not saying much. Story is a confusing mess about corporate greed leading to...


Pred: unk
Texto: While this isn't one of Miss Davies' very worst films, it is pretty bad. And it's sad that

¿Qué pasó?

El modelo no devuelve ninguna palabra “positive” o “negative”, por eso todo cae en "unk".
En Qwen 3-0.6 B, la plantilla de chat agrega tokens de razonamiento internos (<think>) y genera respuestas largas. Ni si quiera forzándolo con un system prompt más estricto y stop tokens se logra clasificar.

Como solución se plantea cambiar a un SLM mas recomendado para la tarea.

### Modelo recomendado (<1B) más estricto para chat

Cambiar a un SLM más obediente y parseá con regex flexible.


In [17]:
!pip -q install "transformers>=4.44" accelerate bitsandbytes

import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype=dtype
)

def chat(system, user, max_new_tokens=32, temperature=0.0, do_sample=False):
    msgs = []
    if system:
        msgs.append({"role":"system","content":system})
    msgs.append({"role":"user","content":user})
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=do_sample, temperature=temperature,
                         pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return gen.strip()


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Clasificador robusto y truncado:

In [18]:
def classify_review(text):
    # limitar longitud para evitar desvíos
    enc = tok(text, truncation=True, max_length=256, return_tensors=None)
    text_trunc = tok.decode(enc["input_ids"], skip_special_tokens=True)

    system = ("Eres un clasificador de sentimiento. "
              "Responde SOLO una palabra EXACTA: positive o negative.")
    user = f'Review:\n"""{text_trunc}"""\nEtiqueta:'

    out = chat(system, user, max_new_tokens=4, temperature=0.0, do_sample=False)
    out_clean = re.sub(r"<.*?>", "", out).strip().lower()

    m = re.search(r"\b(positive|negative)\b", out_clean, re.I)
    return m.group(1).lower() if m else "unk"


Volvemos a testear con una muestra chiquita:

In [19]:
test_df = df_sample.head(5).copy()
test_df["pred_llm"] = test_df["review"].apply(classify_review)
print(test_df[["sentiment","pred_llm"]])


      sentiment  pred_llm
28350  negative  negative
17653  negative  negative
44849  negative  negative
24255  negative  negative
9797   negative  negative


Ahora si funciona!

Pasamos las 500 filas:

In [20]:
# Inferimos
preds = [classify_review(t) for t in tqdm(df_sample["review"], total=len(df_sample))]
df_sample["pred_llm"] = preds

# Evaluamos
from sklearn.metrics import classification_report
print(classification_report(df_sample["sentiment"], df_sample["pred_llm"]))

100%|██████████| 500/500 [01:19<00:00,  6.28it/s]

              precision    recall  f1-score   support

    negative       0.86      0.90      0.88       250
    positive       0.89      0.85      0.87       250

    accuracy                           0.88       500
   macro avg       0.88      0.88      0.88       500
weighted avg       0.88      0.88      0.88       500



### Interpretación

El SLM Qwen2.5-0.5B-Instruct, sin ningún entrenamiento adicional (zero-shot) logra un desempeño alto sin fine-tuning.
Un modelo pequeño (<1 B) alcanza 88 % de acierto en un dataset de reseñas (IMDB-like).
Esto confirma que los SLMs modernos aprenden patrones semánticos y sentimentales durante su pre-entrenamiento masivo.

Las métricas por clase son similares, por lo que el modelo no está sesgado hacia “positive” ni “negative”.

Tiene por ventaje que no requiere etiquetas ni pipeline de features pero el costo computacional resultó ser alto.

La conclusión para este punto es que se valida el uso de SLMs como baseline universal en tareas de NLP incluso sin ajuste fino.

### Comparar el modelo clásico vs SLM

Se quiere comparar el rendimiento del modelo usado en el TP1 (una regresión logística aplicando TF - IDF) contra el SLM de Qwen.

### Reconstrucción y entrenamiento del modelo clásico

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pandas as pd

# usamos el dataset completo
df_base = df
X_train, X_test, y_train, y_test = train_test_split(
    df_base["review"], df_base["sentiment"], test_size=0.2, random_state=0, stratify=df_base["sentiment"]
)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1,2))),
    ("clf", LogisticRegression(max_iter=1000, C=1.0, penalty="l2"))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

    negative       0.91      0.89      0.90      5000
    positive       0.89      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



### Evaluar el mismo modelo sobre la muestra de 500

In [22]:
y_pred_sample = pipe.predict(df_sample["review"])
df_sample["pred_baseline"] = y_pred_sample
print(classification_report(df_sample["sentiment"], df_sample["pred_baseline"]))


              precision    recall  f1-score   support

    negative       0.92      0.88      0.90       250
    positive       0.89      0.92      0.90       250

    accuracy                           0.90       500
   macro avg       0.90      0.90      0.90       500
weighted avg       0.90      0.90      0.90       500



### Comparar ambos modelos cuantitativamente

In [23]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

acc_llm = accuracy_score(df_sample["sentiment"], df_sample["pred_llm"])
acc_base = accuracy_score(df_sample["sentiment"], df_sample["pred_baseline"])
f1_llm = f1_score(df_sample["sentiment"], df_sample["pred_llm"], pos_label="positive")
f1_base = f1_score(df_sample["sentiment"], df_sample["pred_baseline"], pos_label="positive")

pd.DataFrame({
    "modelo": ["TF-IDF + LogisticRegression", "Qwen2.5-0.5B-Instruct (zero-shot)"],
    "accuracy": [acc_base, acc_llm],
    "f1_positive": [f1_base, f1_llm]
})


,modelo,accuracy,f1_positive
0,TF-IDF + LogisticRegression,0.902,0.904110
1,Qwen2.5-0.5B-Instruct (zero-shot),0.876,0.872951


### Interpretación

* El modelo clásico obtiene mejor puntuación porque fue entrenado específicamente sobre el mismo dominio.
Ajusta pesos TF-IDF y coeficientes logísticos para maximizar la separación entre reseñas positivas y negativas.
Se beneficia del conocimiento exacto del vocabulario del dataset.

* El SLM Qwen 0.5 B, aunque no fue entrenado ni fine-tuneado, alcanza ~ 0.88 de accuracy.
Esto prueba que un Small Language Model ya contiene representaciones semánticas ricas aprendidas en pre-entrenamiento, suficientes para resolver una tarea supervisada sin usar etiquetas (zero-shot learning).

En términos de costo-beneficio, el modelo clásico es más eficiente y rápido mientras que el SLM es más versátil y adaptable a nuevos dominios donde no se dispone de datos etiquetados.

## Iteración de parámetros y prompts para lograr mejoras:


Definimos distintos prompts para clasificar. Alternamos entre zero_shots y few_shots para comparar.

In [33]:
PROMPTS = {
    "zs_strict_en": (
        "You are a sentiment classifier. Reply with EXACTLY one word: positive or negative. "
        "No punctuation, no explanations.",
        'Review:\n"""{text}"""\nLabel:'
    ),
    "zs_strict_es": (
        "Eres un clasificador de sentimiento. Responde con UNA sola palabra EXACTA: positive o negative. "
        "Sin puntuación ni explicación.",
        'Reseña:\n"""{text}"""\nEtiqueta:'
    ),
    "fs_min": (
        "Classify sentiment. Output only: positive or negative.",
        "Examples:\n"
        "Text: \"me encantó la película\" -> positive\n"
        "Text: \"fue una pérdida de tiempo\" -> negative\n"
        "Now classify:\n\"\"\"{text}\"\"\"\nLabel:"
    ),
    "zs_calibrated": (
        "You are a strict classifier. Prior over classes is equal. If uncertain, still choose exactly one "
        "from: positive, negative. Any other output is invalid.",
        'Text:\n"""{text}"""\nReturn only one of [positive, negative]:'
    ),
}


In [38]:
LABELS = ["positive","negative"]
LABELS_SET = set(LABELS)

def parse_label(txt):
    if not isinstance(txt, str):
        return "unk"
    t = re.sub(r"<.*?>", "", txt).strip().lower()
    m = re.search(r"\b(positive|negative)\b", t)
    return m.group(1) if m else "unk"

def normalize_pred(raw):
    lab = parse_label(raw)
    return lab if lab in LABELS_SET else "unk"

Además de alternar entre diferentes prompts, sugerimos dos modos de generación:
1. Greedy (determinístico pero con reducción de creatividad )
2. Sampling (Permite mayor creatividad pero puede incorporar ruido)

Y también recortamos la longitud de salida a 1 token.


In [45]:
def predict_one(txt, sys_t, user_t, mx=1, temp=0.0, sample=False, seed=42):
    if sample:
        torch.manual_seed(seed)
    out = chat(sys_t, user_t.format(text=txt),
               max_new_tokens=mx, temperature=temp, do_sample=sample)
    return normalize_pred(out)

def eval_prompts_min(df_text, y_true, prompts):
    rows = []
    X = list(df_text); y = list(y_true)
    setups = [
        ("greedy", 0.0, False, 1),
        ("sample", 0.15, True, 1),
        ("sample", 0.35, True, 1),
    ]
    for pname, (sys_t, user_t) in prompts.items():
        for mode, t, samp, mx in setups:
            preds = [predict_one(x, sys_t, user_t, mx=mx, temp=t, sample=samp) for x in X]
            rows.append(dict(
                prompt=pname, mode=mode, temp=t, max_new_tokens=mx,
                accuracy=accuracy_score(y, preds),
                f1_positive=f1_score(y, preds, pos_label="positive"),
                f1_macro=f1_score(y, preds, average="macro"),
            ))
    res = pd.DataFrame(rows).sort_values(["f1_macro","accuracy"], ascending=False).reset_index(drop=True)
    return res

grid_res = eval_prompts_min(df_sample["review"], df_sample["sentiment"], PROMPTS)
grid_res


,prompt,mode,temp,max_new_tokens,accuracy,f1_positive,f1_macro
0,zs_strict_es,greedy,0.00,1,0.932,0.934866,0.931868
1,zs_strict_es,sample,0.15,1,0.932,0.934866,0.931868
2,zs_strict_es,sample,0.35,1,0.932,0.934866,0.931868
3,fs_min,greedy,0.00,1,0.928,0.930769,0.927885
4,fs_min,sample,0.15,1,0.928,0.930769,0.927885
5,fs_min,sample,0.35,1,0.928,0.930769,0.927885
6,zs_strict_en,greedy,0.00,1,0.920,0.922481,0.919918
7,zs_strict_en,sample,0.15,1,0.920,0.922481,0.919918
8,zs_strict_en,sample,0.35,1,0.920,0.922481,0.919918
9,zs_calibrated,greedy,0.00,1,0.734,0.647215,0.716866


Observamos que la combinación con mejores resultados fue:

__zs_strict_es + greedy + max_new_tokens=1__

Procedemos a probar la clasificación:

In [46]:
from sklearn.metrics import confusion_matrix

BEST_PROMPT = "zs_strict_es"
sys_t, user_t = PROMPTS[BEST_PROMPT]

def classify_review_best(text):
    #enc = tok(text, truncation=True, max_length=256, return_tensors=None)
    #text_trunc = tok.decode(enc["input_ids"], skip_special_tokens=True)

    out = chat(
        sys_t,
        user_t.format(text=text),
        max_new_tokens=1,        # salida 1 token
        temperature=0.0,         # greedy
        do_sample=False
    )
    # normalización a 'positive'/'negative' o 'unk'
    return normalize_pred(out)

X = df_sample["review"].tolist()
y = df_sample["sentiment"].tolist()

preds = [classify_review_best(t) for t in X]

acc  = accuracy_score(y, preds)
f1p  = f1_score(y, preds, pos_label="positive")
f1m  = f1_score(y, preds, average="macro")
cm   = confusion_matrix(y, preds, labels=["positive","negative"])

print("Accuracy:", acc)
print("F1 positive:", f1p)
print("F1 macro:", f1m)
print("\nReporte:\n", classification_report(y, preds))

pd.DataFrame(cm, index=["true_pos","true_neg"], columns=["pred_pos","pred_neg"])


Accuracy: 0.932
F1 positive: 0.9348659003831418
F1 macro: 0.9318680966350855

Reporte:
               precision    recall  f1-score   support

    negative       0.97      0.89      0.93       250
    positive       0.90      0.98      0.93       250

    accuracy                           0.93       500
   macro avg       0.94      0.93      0.93       500
weighted avg       0.94      0.93      0.93       500



,pred_pos,pred_neg
true_pos,244,6
true_neg,28,222


Conclusión:

La principal mejora en el rendimiento del modelo se logró al eliminar el truncado temprano de las reseñas (256 tokens), que estaba descartando información clave para la clasificación. Esta modificación permitió que el modelo tuviera acceso completo a las reseñas, lo cual se reflejó directamente en un aumento de las métricas.

Por otro lado, el ajuste de hiperparámetros del modelo SLM mediante una búsqueda en grid también contribuyó significativamente. Esta combinación (input completo + tuning adecuado) permitió alcanzar un F1-score de aproximadamente 0.932, superando al modelo tradicional basado en TF-IDF.

Estos resultados indican que, en este caso, un modelo SLM bien configurado no solo puede competir, sino incluso superar a métodos clásicos, al aprovechar mejor la semántica implícita en los textos completos.